# Hierarchical Clustering with EVōC

Fast, multi-layer embedding clustering using EVōC. Produces hierarchical clusters and quality metrics automatically.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
from __future__ import annotations
import csv
import hashlib
import html
import json
import os
import pickle
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")          # keep Colab output clean

# ── Third-party (always available) ───────────────────────────────────────────
import numpy as np
import torch
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer

# ── EVoC for clustering ───────────────────────────────────────────────────────
try:
    import evoc
    EVOC_AVAILABLE = True
except ImportError:
    EVOC_AVAILABLE = False
    print("⚠ EVoC not installed. Run: pip install evoc")

In [ ]:
# ── GPU / CPU detection ───────────────────────────────────────────────────────
CUDA_AVAILABLE = torch.cuda.is_available()

if CUDA_AVAILABLE:
    print("✓ GPU detected")
else:
    print("ℹ CPU mode")

# ── Constants ─────────────────────────────────────────────────────────────────
CSV_FILE = Path("/content/train.csv")

_DRIVE_CACHE = "/content/drive/MyDrive/clustering_cache_evoc"
_LOCAL_CACHE = "/tmp/clustering_cache_evoc"

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Data loading and cleaning
# ─────────────────────────────────────────────────────────────────────────────

# Compiled once at module load — cheap to reuse
_RE_SOURCE_SUFFIX = re.compile(r'\s+-\s+\S+\.\S+$')   # " - www.site.com" at end of title
_RE_HTML_TAG      = re.compile(r'<[^>]+>')
_RE_NONALPHA      = re.compile(r'[^a-zA-Z0-9\s\'\'\-]')
_RE_SPACES        = re.compile(r'\s+')
_RE_SHORT_NUM     = re.compile(r'(?<![a-zA-Z0-9\-])\d{1,3}(?![a-zA-Z0-9\-])')
_RE_THOUSANDS     = re.compile(r'(\d),(\d)')


def clean_title(raw: str) -> str:
    """Normalise a news headline for both TF-IDF and embedding."""
    s = html.unescape(raw)                  # first pass: &amp;#39; → &#39;
    s = html.unescape(s)                    # second pass: &#39; → '
    s = _RE_SOURCE_SUFFIX.sub('', s)        # "Title - site.com" → "Title"
    s = _RE_HTML_TAG.sub(' ', s)            # <b>foo</b> → foo
    s = _RE_THOUSANDS.sub(r'\1\2', s)      # "1,000" → "1000" before comma-strip
    s = _RE_THOUSANDS.sub(r'\1\2', s)      # second pass for "1,000,000"
    s = _RE_NONALPHA.sub(' ', s)            # remove remaining punctuation / symbols
    s = _RE_SHORT_NUM.sub(' ', s)           # remove bare 1-3 digit tokens (entity residue)
    s = _RE_SPACES.sub(' ', s)              # collapse whitespace
    return s.strip()


def load_titles_from_csv(csv_file: Path = CSV_FILE) -> list[str]:
    """Load and clean non-empty values from the 'Title' column of a CSV file."""
    titles: list[str] = []
    with csv_file.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw = (row.get("Title") or "").strip()
            if not raw:
                continue
            cleaned = clean_title(raw)
            if cleaned:
                titles.append(cleaned)
    return titles

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cache helpers
# ─────────────────────────────────────────────────────────────────────────────

def _resolve_cache_dir() -> str:
    """Return the cache directory to use."""
    if os.path.isdir("/content/drive/MyDrive"):
        cache_dir = _DRIVE_CACHE
    else:
        cache_dir = _LOCAL_CACHE
    os.makedirs(cache_dir, exist_ok=True)
    return cache_dir


CACHE_DIR = _resolve_cache_dir()


def _cache_path(name: str) -> str:
    return os.path.join(CACHE_DIR, f"{name}.pkl")


def save_cache(name: str, obj) -> None:
    path = _cache_path(name)
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_cache(name: str):
    path = _cache_path(name)
    if os.path.exists(path):
        with open(path, "rb") as f:
            obj = pickle.load(f)
        return obj
    return None


def data_fingerprint(sentences: list[str]) -> str:
    """Compute MD5 hash of cleaned sentences for cache invalidation."""
    content = "\n".join(sentences)
    return hashlib.md5(content.encode()).hexdigest()[:8]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Topic inference using TF-IDF
# ─────────────────────────────────────────────────────────────────────────────

def infer_topic(titles: list[str], top_n: int = 3) -> str:
    """Extract top N TF-IDF terms as a topic string."""
    if not titles or len(titles) < 2:
        return "Cluster"

    try:
        vectorizer = TfidfVectorizer(max_features=100, stop_words="english")
        _ = vectorizer.fit_transform(titles)
        feature_names = vectorizer.get_feature_names_out()
        if len(feature_names) > 0:
            return " ".join(feature_names[:top_n])
    except Exception:
        pass

    return "Cluster"

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Quality metrics
# ─────────────────────────────────────────────────────────────────────────────

def _compute_silhouette(embeddings: np.ndarray, labels: np.ndarray) -> float:
    """Compute silhouette score (range -1 to 1, higher is better)."""
    # Remove noise points (label == -1)
    mask = labels != -1
    if mask.sum() < 2:
        return 0.0
    try:
        return silhouette_score(
            embeddings[mask],
            labels[mask],
            sample_size=min(5000, mask.sum())
        )
    except Exception:
        return 0.0


def _cluster_size_stats(labels: np.ndarray) -> dict:
    """Compute cluster size statistics, excluding noise."""
    counts = Counter(labels[labels != -1])
    if not counts:
        return {"min": 0, "max": 0, "median": 0, "mean": 0.0, "std": 0.0}
    sizes = np.array(list(counts.values()))
    return {
        "min": int(sizes.min()),
        "max": int(sizes.max()),
        "median": int(np.median(sizes)),
        "mean": float(sizes.mean()),
        "std": float(sizes.std()),
    }


def _noise_percentage(labels: np.ndarray) -> float:
    """Percentage of points labeled as noise (-1)."""
    return 100.0 * (labels == -1).sum() / len(labels)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Main pipeline with EVoC
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    if not EVOC_AVAILABLE:
        raise ImportError("EVoC is required. Install with: pip install evoc")

    # ── 0. Load data ──────────────────────────────────────────────────────────
    sentences = load_titles_from_csv()
    if len(sentences) < 2:
        raise ValueError("Need at least 2 titles to cluster.")

    N = len(sentences)
    fp = data_fingerprint(sentences)
    print(f"\n{'═'*50}")
    print(f"CLUSTERING PIPELINE (EVoC)")
    print(f"{'═'*50}")
    print(f"Dataset: {N:,} titles")
    print(f"Cache:   {CACHE_DIR}\n")

    # ── 1. Embeddings ─────────────────────────────────────────────────────────
    emb_key = f"embeddings_{fp}"
    embeddings = load_cache(emb_key)

    if embeddings is None:
        device = "cuda" if CUDA_AVAILABLE else "cpu"
        print(f"[1] Computing embeddings ({device}) …")
        model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)
        embeddings = model.encode(
            sentences,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=64 if CUDA_AVAILABLE else 16,
            show_progress_bar=True,
        )
        del model
        if CUDA_AVAILABLE:
            torch.cuda.empty_cache()
        save_cache(emb_key, embeddings)
    else:
        print(f"[1] Embeddings loaded from cache")

    # ── 2. EVoC hierarchical clustering ───────────────────────────────────────
    print(f"[2] EVoC hierarchical clustering …")
    evoc_key = f"evoc_clusterer_{fp}"
    evoc_model = load_cache(evoc_key)

    fit_labels = None
    if evoc_model is None:
        evoc_model = evoc.EVoC()
        fit_labels = evoc_model.fit_predict(embeddings)
        save_cache(evoc_key, evoc_model)
    
    cluster_layers = getattr(evoc_model, "cluster_layers_", None)
    if cluster_layers is None:
        cluster_layers = getattr(evoc_model, "layers_", None)

    cluster_labels = getattr(evoc_model, "cluster_labels_", None)
    if cluster_labels is None:
        cluster_labels = getattr(evoc_model, "labels_", None)
    if cluster_labels is None and cluster_layers is not None and len(cluster_layers) > 0:
        cluster_labels = cluster_layers[0]
    if cluster_labels is None and fit_labels is not None:
        cluster_labels = fit_labels
    if cluster_layers is None and cluster_labels is not None:
        cluster_layers = [np.asarray(cluster_labels)]

    cluster_tree = getattr(evoc_model, "cluster_tree_", None)
    if cluster_tree is None:
        cluster_tree = getattr(evoc_model, "tree_", None)

    potential_duplicates = getattr(evoc_model, "duplicates_", None)
    if potential_duplicates is None:
        potential_duplicates = getattr(evoc_model, "duplicate_groups_", None)
    if potential_duplicates is None:
        potential_duplicates = []

    if cluster_labels is None or cluster_layers is None:
        available = [a for a in dir(evoc_model) if a.endswith("_")]
        raise AttributeError(
            "EVoC output attributes not found. Available attrs: " + ", ".join(sorted(available))
        )

    # Quality metrics for finest-grained clustering
    sil = _compute_silhouette(embeddings, cluster_labels)
    sizes = _cluster_size_stats(cluster_labels)
    noise_pct = _noise_percentage(cluster_labels)
    n_clusters = len(set(cluster_labels) - {-1})

    print(f"    → {n_clusters} clusters (layer 0 / finest)")
    print(f"    └ Quality: silhouette={sil:.2f}, noise={noise_pct:.1f}%")
    print(f"    └ Sizes: {sizes['min']}–{sizes['max']} (median {sizes['median']})")

    # ── 3. Infer topics per cluster ───────────────────────────────────────────
    print(f"[3] Inferring topics …")
    topics_dict: dict[int, str] = {}
    for cluster_id in sorted(set(cluster_labels) - {-1}):
        mask = cluster_labels == cluster_id
        cluster_sents = [sentences[i] for i in np.where(mask)[0]]
        topics_dict[cluster_id] = infer_topic(cluster_sents)

    print(f"    → {len(topics_dict)} topics extracted")

    # ── 4. Layer statistics ──────────────────────────────────────────────────
    print(f"[4] Cluster layer hierarchy:")
    for layer_idx, layer_labels in enumerate(cluster_layers):
        n_layer_clusters = len(set(layer_labels) - {-1})
        layer_sizes = _cluster_size_stats(layer_labels)
        layer_noise = _noise_percentage(layer_labels)
        print(f"    Layer {layer_idx}: {n_layer_clusters:3d} clusters, ",
              f"noise={layer_noise:.1f}%, ",
              f"sizes {layer_sizes['min']}–{layer_sizes['max']}")

    # ── 5. Save results ──────────────────────────────────────────────────────
    print(f"[5] Saving results …")
    results_key = f"evoc_results_{fp}"
    results = {
        "sentences": sentences,
        "embeddings": embeddings,
        "cluster_labels": cluster_labels,
        "cluster_layers": cluster_layers,
        "cluster_tree": cluster_tree,
        "topics_dict": topics_dict,
        "duplicates": potential_duplicates,
    }
    save_cache(results_key, results)

    print(f"\n{'═'*50}")
    print(f"✓ PIPELINE COMPLETE")
    print(f"{'═'*50}\n")
    print(f"Results cached as: {results_key}")
    return results

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cache status
# ─────────────────────────────────────────────────────────────────────────────
import glob
existing = glob.glob(f"{CACHE_DIR}/*.pkl") + glob.glob(f"{CACHE_DIR}/*.npz") + glob.glob(f"{CACHE_DIR}/*.json")
if existing:
    total_mb = sum(os.path.getsize(p)/1e6 for p in existing)
    print(f"\nCache: {len(existing)} files ({total_mb:.1f} MB)")
else:
    print(f"\nCache: empty (will compute from scratch)")

In [ ]:
# Run the pipeline
results = main()

## Explore Results

EVoC provides multiple granularities of clusters. Use these cells to explore the hierarchy.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Compare cluster sizes across layers
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("CLUSTER SIZE DISTRIBUTION ACROSS LAYERS")
print("="*60)

cluster_labels = results["cluster_labels"]
cluster_layers = results["cluster_layers"]

for layer_idx, layer_labels in enumerate(cluster_layers):
    counts = Counter(layer_labels[layer_labels != -1])
    if not counts:
        continue
    sizes = sorted(counts.values())
    print(f"\nLayer {layer_idx}: {len(counts)} clusters")
    print(f"  Size range: {min(sizes)} → {max(sizes)}")
    print(f"  Mean size: {np.mean(sizes):.1f} (±{np.std(sizes):.1f})")
    print(f"  Noise points: {(layer_labels == -1).sum()}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Show sample clusters with topics
# ─────────────────────────────────────────────────────────────────────────────

sentences = results["sentences"]
cluster_labels = results["cluster_labels"]
topics_dict = results["topics_dict"]

print("\n" + "="*60)
print("SAMPLE CLUSTERS (TOP LAYER)")
print("="*60)

for cluster_id in sorted(set(cluster_labels) - {-1})[:10]:  # Show first 10
    mask = cluster_labels == cluster_id
    cluster_sents = [sentences[i] for i in np.where(mask)[0]]
    topic = topics_dict.get(cluster_id, "Unknown")

    print(f"\n[{cluster_id}] {topic} ({len(cluster_sents)} items)")
    for sent in cluster_sents[:3]:
        print(f"    • {sent}")
    if len(cluster_sents) > 3:
        print(f"    … and {len(cluster_sents) - 3} more")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Inspect duplicates if detected
# ─────────────────────────────────────────────────────────────────────────────

duplicates = results["duplicates"]

if duplicates:
    print("\n" + "="*60)
    print(f"POTENTIAL DUPLICATES ({len(duplicates)} groups)")
    print("="*60)

    for i, group in enumerate(duplicates[:5]):
        print(f"\nDuplicate group {i}:")
        for idx in group[:3]:
            print(f"  {sentences[idx]}")
        if len(group) > 3:
            print(f"  … and {len(group) - 3} more")
else:
    print("\nNo duplicates detected.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Optional: Create export JSON for downstream tools
# ─────────────────────────────────────────────────────────────────────────────

import json

# Create a JSON export of cluster structure
cluster_labels = results["cluster_labels"]
topics_dict = results["topics_dict"]
sentences = results["sentences"]

export_data = {
    "metadata": {
        "total_documents": len(sentences),
        "total_clusters": len(topics_dict),
        "embedding_model": "BAAI/bge-large-en-v1.5",
        "clustering_method": "EVoC",
    },
    "clusters": []
}

for cluster_id in sorted(set(cluster_labels) - {-1}):
    mask = cluster_labels == cluster_id
    indices = np.where(mask)[0].tolist()
    cluster_sents = [sentences[i] for i in indices]
    
    export_data["clusters"].append({
        "id": int(cluster_id),
        "topic": topics_dict.get(cluster_id, "Unknown"),
        "size": len(cluster_sents),
        "documents": cluster_sents[:5],  # Show first 5
    })

export_path = os.path.join(CACHE_DIR, "cluster_export.json")
with open(export_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print(f"✓ Exported to: {export_path}")